# P91 — Fusión, propagación y estructuración en redes de creencia

## 1. Título y paper

**Paper:** *Fusion, Propagation, and Structuring in Belief Networks*  
**Autoría:** Judea Pearl  
**Año y venue:** 1986 · Artificial Intelligence, 29(3), 241–288  
**Nivel:** L3 · **Motor:** `redes_bayesianas`  
**Ficha completa:** [`P91_redes_bayesianas`](../../papers/foundational/P91_redes_bayesianas/README.md)

**Hito:** Hace tratable la probabilidad en IA: la estructura del grafo dice qué hay que almacenar y qué se puede propagar localmente.

- [doi:10.1016/0004-3702(86)90072-X](https://doi.org/10.1016/0004-3702%2886%2990072-X)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: Aplicar probabilidad a un dominio con decenas de variables exige una tabla conjunta con 2ⁿ entradas: imposible de almacenar, de estimar y de actualizar. Esa fue la razón técnica por la que la IA de los setenta la abandonó en favor de los factores de certeza.
2. Ejecutar una implementación mínima de la propuesta: Representar las dependencias con un grafo dirigido acíclico. Las independencias condicionales que el grafo codifica reducen la conjunta a un producto de condicionales locales, y permiten propagar creencias por paso de mensajes entre nodos vecinos.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P87
- P88
- P69


## 4. Intuición

Con 30 variables binarias, la tabla de probabilidad conjunta tiene mil millones de entradas. Nadie la puede estimar ni almacenar, y por eso la IA de los setenta abandonó la probabilidad. Pearl la recupera con una observación: casi todas esas entradas son redundantes, porque casi todo es condicionalmente independiente de casi todo.


## 5. Concepto mínimo

```text
Grafo dirigido acíclico + una tabla condicional por nodo

    P(x₁…xₙ) = Π P(xᵢ | padres(xᵢ))

Y dos patrones que el grafo codifica y la asociación no:
    causa común    : A ← C → B   son dependientes, e independientes DADO C
    efecto común   : A → E ← B   son independientes, y DEPENDIENTES dado E
                     (explicar y descartar)
```


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('redes_bayesianas', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. ¿Cuánto sube la probabilidad de lluvia al ver el césped mojado?
2. ¿Y si además sabemos que el aspersor estuvo encendido?
3. ¿Cuántos parámetros ahorra el grafo?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('redes_bayesianas', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('redes_bayesianas', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

Ver el césped mojado sube la lluvia a **0,7079**. Saber además que el aspersor estuvo encendido la **baja** a **0,3204**: una causa explica el efecto y descarta a la otra. Y la tabla conjunta completa exige 15 parámetros frente a los 9 de la red — con 4 variables la diferencia es pequeña; con 30 son mil millones frente a unas decenas.


## 10. Comentario pedagógico

«Explicar y descartar» es la firma del razonamiento causal y ningún modelo puramente asociativo la produce: dos variables independientes se vuelven dependientes al observar su efecto común. Esa asimetría entre grafo y correlación es lo que años después [P95](../../papers/foundational/P95_causalidad/README.md) convierte en una escalera de tres peldaños.


## 11. Error o anti-patrón deliberado

Anti-patrón: leer las flechas del grafo como si fueran correlaciones.


In [ ]:
print('Aspersor y lluvia son INDEPENDIENTES a priori... si no sabes nada mas.')
print('Al observar el cesped mojado se vuelven dependientes: explicar y descartar.')
print('Condicionar sobre un efecto comun CREA dependencia. Eso no lo hace ninguna correlacion.')

## 12. Corrección

Los dos patrones, medidos sobre la misma red:


In [ ]:
r = run_paper_lab('redes_bayesianas', seed=7)['result']
print('P(lluvia | mojado)            =', r['P_lluvia_dado_mojado'])
print('P(lluvia | mojado, aspersor)  =', r['P_lluvia_dado_mojado_y_aspersor'])
print('independencia condicional:', r['independencia_condicional_se_cumple'])
print('parametros: conjunta', r['parametros_tabla_conjunta_completa'],
      'vs red', r['parametros_de_la_red'])

## 13. Desafío guiado

Comprueba en la salida que lluvia y aspersor son independientes dado «nublado», y explica por qué eso es lo que ahorra parámetros.


In [ ]:
r = run_paper_lab('redes_bayesianas', seed=3)['result']
show(r)

## 14. Desafío autónomo

Modela con una red bayesiana un diagnóstico de tu dominio: cuatro o cinco variables, con su grafo y sus tablas. Después calcula una consulta con evidencia parcial y contrasta el resultado con tu intuición.


## 15. Evidencia de aprendizaje

Guarda las dos consultas —con y sin la información del aspersor— y tu explicación de explicar y descartar.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P91_redes_bayesianas/README.md) · evaluación formal: [`assessments/papers/P91_redes_bayesianas.md`](../../assessments/papers/P91_redes_bayesianas.md)


## 16. Cierre

El grafo hace tratable la inferencia. Volvemos a la búsqueda: dos familias que no recombinan soluciones sino que comparten información mientras exploran.


## 17. Conexión con el siguiente hito

- P95
- P94

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
